In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive_output, FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, Output
from IPython.display import display

# ------------------------------------------------------------
# 1. DESCRIPTION
# ------------------------------------------------------------

description = HTML("""
<div style="
    border:1px solid #b9d7f5;
    border-radius:8px;
    padding:8px 10px;
    margin-bottom:10px;
    font-size:13px;
    line-height:1.35;
    background-color:#f7fbff;
">
<div><b>Purpose:</b> Explore the component-level behavior of second-order multiple-feedback active filters.</div>
<div><b>Available filters:</b> Low-pass, high-pass, band-pass, and band-stop.</div>
<div><b>What we see:</b> The magnitude and phase responses together with the values of ω₀ and Q calculated directly from the circuit components.</div>
<div><b>What happens as we interact:</b> Changing the resistors and capacitors modifies the filter parameters and immediately changes its frequency response.</div>
</div>
""")

# ------------------------------------------------------------
# 2. FILTER TYPE
# ------------------------------------------------------------

filter_selector = RadioButtons(options=['Low-pass', 'High-pass', 'Band-pass', 'Band-stop'], value='Low-pass', description='', layout=Layout(width='480px', height='26px', margin='0px', padding='0px'))

filter_selector.add_class('horizontal-radio')

radio_style = HTML("""
<style>

.horizontal-radio {
    margin:0 !important;
    padding:0 !important;
    height:26px !important;
}

.horizontal-radio .widget-radio-box {
    display:flex !important;
    flex-direction:row !important;
    flex-wrap:nowrap !important;
    align-items:center !important;
    gap:24px !important;
    margin:0 !important;
    padding:0 !important;
    height:26px !important;
}

.horizontal-radio .widget-radio-box label {
    display:flex !important;
    flex-direction:row !important;
    align-items:center !important;
    margin:0 !important;
    padding:0 !important;
    height:26px !important;
    line-height:26px !important;
    white-space:nowrap !important;
    font-size:13px !important;
    font-weight:bold !important;
}

.horizontal-radio .widget-radio-box input {
    margin:0 0 0 8px !important;
    padding:0 !important;
    vertical-align:middle !important;
}

.horizontal-radio > label {
    display:none !important;
}

</style>
""")

filter_type_label = HTML("""
<div style="
    display:flex;
    align-items:center;
    justify-content:flex-start;
    height:26px;
    line-height:26px;
    margin:0 25px 0 0;
    padding:0;
    font-size:13px;
    font-weight:bold;
    white-space:nowrap;
">
Filter Type:
</div>
""")

# ------------------------------------------------------------
# 3. CIRCUIT PARAMETERS
# ------------------------------------------------------------

r1_slider = FloatSlider(min=5.0, max=50.0, step=0.5, value=10.0, description='R₁:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))
r2_slider = FloatSlider(min=5.0, max=50.0, step=0.5, value=10.0, description='R₂:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))
r3_slider = FloatSlider(min=5.0, max=50.0, step=0.5, value=10.0, description='R₃:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))
r4_slider = FloatSlider(min=0.1, max=100.0, step=0.1, value=10.0, description='R₄:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))

c1_slider = FloatSlider(min=5.0, max=50.0, step=1.0, value=10.0, description='C₁:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))
c2_slider = FloatSlider(min=5.0, max=50.0, step=1.0, value=10.0, description='C₂:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))
c3_slider = FloatSlider(min=5.0, max=50.0, step=1.0, value=10.0, description='C₃:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))

# ------------------------------------------------------------
# 4. LEGEND AND LABELS
# ------------------------------------------------------------

legend_html = HTML("""
<div style="
    border:1px solid #cccccc;
    border-radius:5px;
    padding:7px 9px;
    width:135px;
    font-size:13px;
    line-height:1.7;
    background:white;
">
<div><span style="display:inline-block; width:32px; border-top:3px solid red; vertical-align:middle; margin-right:7px;"></span>|H(jω)|</div>
<div><span style="display:inline-block; width:32px; border-top:2px dotted black; vertical-align:middle; margin-right:7px;"></span>ω₀</div>
</div>
""")

parameter_label = HTML("<div style='font-size:14px; font-weight:bold; margin-top:12px; margin-bottom:4px;'>Circuit Parameters:</div>")

units_html = HTML("""
<div style="
    border:1px solid #cccccc;
    border-radius:5px;
    padding:7px 9px;
    margin-top:7px;
    font-size:12px;
    line-height:1.6;
    color:#555555;
    background:white;
    width:110px;
">
R₁, R₂, R₃, R₄ in kΩ<br>
C₁, C₂, C₃ in nF
</div>
""")

# ------------------------------------------------------------
# 5. OUTPUT AREAS
# ------------------------------------------------------------

magnitude_output = Output(layout=Layout(width='100%', overflow='hidden'))
phase_output = Output(layout=Layout(width='100%', overflow='hidden'))
info_output = Output(layout=Layout(width='auto', overflow='hidden'))

# ------------------------------------------------------------
# 6. MAIN UPDATE FUNCTION
# ------------------------------------------------------------

def update_multiple_feedback(filter_type, R1, R2, R3, R4, C1, C2, C3):

    R1_si = R1 * 1e3
    R2_si = R2 * 1e3
    R3_si = R3 * 1e3
    R4_si = R4 * 1e3

    C1_si = C1 * 1e-9
    C2_si = C2 * 1e-9
    C3_si = C3 * 1e-9

    # --------------------------------------------------------
    # LOW-PASS MULTIPLE FEEDBACK
    # --------------------------------------------------------

    if filter_type == 'Low-pass':

        omega0_squared = 1.0 / (R2_si * R3_si * C1_si * C2_si)
        omega0 = np.sqrt(omega0_squared)

        a1 = (1.0 / R1_si + 1.0 / R2_si + 1.0 / R3_si) / C2_si

        Q = omega0 / a1

        reference_name = 'DC gain'
        reference_gain = R2_si / R1_si

        used_components = f"""
        <div><b>R₁:</b> <span style="color:#0066cc;">{R1:.1f} kΩ</span></div>
        <div><b>R₂:</b> <span style="color:#0066cc;">{R2:.1f} kΩ</span></div>
        <div><b>R₃:</b> <span style="color:#0066cc;">{R3:.1f} kΩ</span></div>
        <div><b>C₁:</b> <span style="color:#0066cc;">{C1:.1f} nF</span></div>
        <div><b>C₂:</b> <span style="color:#0066cc;">{C2:.1f} nF</span></div>
        """

    # --------------------------------------------------------
    # HIGH-PASS MULTIPLE FEEDBACK
    # --------------------------------------------------------

    elif filter_type == 'High-pass':

        omega0_squared = 1.0 / (R1_si * R2_si * C2_si * C3_si)
        omega0 = np.sqrt(omega0_squared)

        Q = np.sqrt(R1_si / R2_si) / (np.sqrt(C3_si / C2_si) + np.sqrt(C2_si / C3_si) + C1_si / np.sqrt(C2_si * C3_si))

        a1 = omega0 / Q

        reference_name = 'High-frequency gain'
        reference_gain = C1_si / C2_si

        used_components = f"""
        <div><b>R₁:</b> <span style="color:#0066cc;">{R1:.1f} kΩ</span></div>
        <div><b>R₂:</b> <span style="color:#0066cc;">{R2:.1f} kΩ</span></div>
        <div><b>C₁:</b> <span style="color:#0066cc;">{C1:.1f} nF</span></div>
        <div><b>C₂:</b> <span style="color:#0066cc;">{C2:.1f} nF</span></div>
        <div><b>C₃:</b> <span style="color:#0066cc;">{C3:.1f} nF</span></div>
        """

    # --------------------------------------------------------
    # BAND-PASS MULTIPLE FEEDBACK
    # --------------------------------------------------------

    elif filter_type == 'Band-pass':

        omega0_squared = (R1_si + R2_si) / (R1_si * R2_si * R3_si * C1_si * C2_si)
        omega0 = np.sqrt(omega0_squared)

        a1 = (1.0 / C1_si + 1.0 / C2_si) / R3_si

        Q = omega0 / a1

        reference_name = 'Center gain'

        used_components = f"""
        <div><b>R₁:</b> <span style="color:#0066cc;">{R1:.1f} kΩ</span></div>
        <div><b>R₂:</b> <span style="color:#0066cc;">{R2:.1f} kΩ</span></div>
        <div><b>R₃:</b> <span style="color:#0066cc;">{R3:.1f} kΩ</span></div>
        <div><b>C₁:</b> <span style="color:#0066cc;">{C1:.1f} nF</span></div>
        <div><b>C₂:</b> <span style="color:#0066cc;">{C2:.1f} nF</span></div>
        """

    # --------------------------------------------------------
    # BAND-STOP MULTIPLE FEEDBACK
    # --------------------------------------------------------

    else:

        # R4 is determined by the notch condition:
        #
        # 1/(R2*C1) + 1/(R2*C2) - R3/(R1*R4*C1) = 0
        #
        # Therefore:
        #
        # R4 = R2*R3*C2 / [R1*(C1+C2)]

        R4_si = (R2_si * R3_si * C2_si) / (R1_si * (C1_si + C2_si))
        R4_auto = R4_si / 1e3

        omega0_squared = 1.0 / (R1_si * R2_si * C1_si * C2_si)
        omega0 = np.sqrt(omega0_squared)

        a1 = 1.0 / (R2_si * C1_si) + 1.0 / (R2_si * C2_si)

        Q = omega0 / a1

        reference_name = 'Asymptotic gain'
        reference_gain = R4_si / (R3_si + R4_si)

        used_components = f"""
        <div><b>R₁:</b> <span style="color:#0066cc;">{R1:.1f} kΩ</span></div>
        <div><b>R₂:</b> <span style="color:#0066cc;">{R2:.1f} kΩ</span></div>
        <div><b>R₃:</b> <span style="color:#0066cc;">{R3:.1f} kΩ</span></div>
        <div><b>R₄ (auto):</b> <span style="color:#0066cc;">{R4_auto:.2f} kΩ</span></div>
        <div><b>C₁:</b> <span style="color:#0066cc;">{C1:.1f} nF</span></div>
        <div><b>C₂:</b> <span style="color:#0066cc;">{C2:.1f} nF</span></div>
        """

    # --------------------------------------------------------
    # FREQUENCY AXIS
    # --------------------------------------------------------

    omega = np.logspace(np.log10(omega0 / 20.0), np.log10(omega0 * 20.0), 4000)

    denominator = (omega0_squared - omega**2) + 1j * a1 * omega

    # --------------------------------------------------------
    # FREQUENCY RESPONSE
    # --------------------------------------------------------

    if filter_type == 'Low-pass':

        numerator = 1.0 / (R1_si * R3_si * C1_si * C2_si)

        H = numerator / denominator

    elif filter_type == 'High-pass':

        numerator = (C1_si / C2_si) * (1j * omega)**2

        H = numerator / denominator

    elif filter_type == 'Band-pass':

        numerator = -(1j * omega) / (R1_si * C2_si)

        H = numerator / denominator

        H0 = -(1j * omega0) / (R1_si * C2_si) / (1j * a1 * omega0)

        reference_gain = np.abs(H0)

    else:

        Kbs = R4_si / (R3_si + R4_si)

        numerator = Kbs * (omega0_squared - omega**2)

        H = numerator / denominator

    magnitude = np.abs(H)
    phase = np.angle(H, deg=True)

    # --------------------------------------------------------
    # MAGNITUDE RESPONSE
    # --------------------------------------------------------

    with magnitude_output:

        magnitude_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 3.5))

        ax.semilogx(omega, magnitude, 'r-', linewidth=2.0)

        ax.axvline(omega0, color='black', linestyle=':', linewidth=1.5)

        if filter_type == 'Band-stop':

            ax.plot(omega0, 0.0, 'ko', markersize=4)

        else:

            magnitude_at_omega0 = np.interp(omega0, omega, magnitude)

            ax.plot(omega0, magnitude_at_omega0, 'ko', markersize=4)

        y_max = max(1.5, min(10.0, np.max(magnitude) * 1.15))

        ax.set_xlim(omega[0], omega[-1])
        ax.set_ylim(0.0, y_max)

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)
        ax.set_ylabel('Magnitude |H(jω)|', fontsize=11)

        ax.set_title(f'Multiple-Feedback Second-Order {filter_type} Filter — Magnitude Response', fontsize=13, fontweight='bold', pad=8)

        ax.grid(True, which='both', linestyle=':', alpha=0.35)

        fig.subplots_adjust(left=0.11, right=0.98, bottom=0.18, top=0.88)

        plt.show()
        plt.close(fig)

    # --------------------------------------------------------
    # PHASE RESPONSE
    # --------------------------------------------------------

    with phase_output:

        phase_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 3.2))

        ax.semilogx(omega, phase, 'r-', linewidth=2.0)

        ax.axvline(omega0, color='black', linestyle=':', linewidth=1.5)

        ax.axhline(0.0, color='gray', linestyle='--', linewidth=1.0)

        ax.set_xlim(omega[0], omega[-1])

        if filter_type == 'Low-pass':

            ax.set_ylim(-190.0, 10.0)
            ax.set_yticks([0, -45, -90, -135, -180])

        elif filter_type == 'High-pass':

            ax.set_ylim(-190.0, 190.0)
            ax.set_yticks([-180, -90, 0, 90, 180])

        elif filter_type == 'Band-pass':

            ax.set_ylim(-190.0, 190.0)
            ax.set_yticks([-180, -90, 0, 90, 180])

        else:

            ax.set_ylim(-100.0, 100.0)
            ax.set_yticks([-90, -45, 0, 45, 90])

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)
        ax.set_ylabel('Phase (degrees)', fontsize=11)

        ax.set_title(f'Multiple-Feedback Second-Order {filter_type} Filter — Phase Response', fontsize=13, fontweight='bold', pad=8)

        ax.grid(True, which='both', linestyle=':', alpha=0.35)

        fig.subplots_adjust(left=0.11, right=0.98, bottom=0.19, top=0.87)

        plt.show()
        plt.close(fig)

    # --------------------------------------------------------
    # INFORMATION FRAME
    # --------------------------------------------------------

    if filter_type == 'Band-stop':

        extra_info = """
        <div>
            <b>Notch:</b>
            <span style="color:#0066cc;">|H(jω₀)| = 0</span>
        </div>
        """

    else:

        extra_info = ""

    info_html = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:6px 9px;
        font-size:12px;
        background:white;
        display:inline-block;
        width:auto;
        box-sizing:border-box;
        white-space:nowrap;
    ">

        <div style="display:flex; align-items:center; gap:22px;">
            <div><b>Filter:</b> <span style="color:#0066cc;">{filter_type}</span></div>
            <div><b>ω₀:</b> <span style="color:#0066cc;">{omega0:.2f} rad/s</span></div>
            <div><b>f₀:</b> <span style="color:#0066cc;">{omega0 / (2.0 * np.pi):.2f} Hz</span></div>
            <div><b>Q:</b> <span style="color:#0066cc;">{Q:.3f}</span></div>
            <div><b>{reference_name}:</b> <span style="color:#0066cc;">{reference_gain:.3f}</span></div>
            {extra_info}
        </div>

        <div style="
            margin-top:4px;
            padding-top:4px;
            border-top:1px solid #eeeeee;
            display:flex;
            align-items:center;
            gap:18px;
        ">
            {used_components}
        </div>

    </div>
    """

    with info_output:

        info_output.clear_output(wait=True)

        display(HTML(info_html))

# ------------------------------------------------------------
# 7. R4 CONTROL FOR BAND-STOP
# ------------------------------------------------------------

def update_r4_state(change):

    if change['new'] == 'Band-stop':
        r4_slider.disabled = True

    else:
        r4_slider.disabled = False

filter_selector.observe(update_r4_state, names='value')

# ------------------------------------------------------------
# 8. INTERACTION
# ------------------------------------------------------------

interactive_controls = interactive_output(update_multiple_feedback, {'filter_type': filter_selector, 'R1': r1_slider, 'R2': r2_slider, 'R3': r3_slider, 'R4': r4_slider, 'C1': c1_slider, 'C2': c2_slider, 'C3': c3_slider})

interactive_controls.layout.display = 'none'

# ------------------------------------------------------------
# 9. LEFT COLUMN
# ------------------------------------------------------------

control_column = VBox([legend_html, parameter_label, r1_slider, r2_slider, r3_slider, r4_slider, c1_slider, c2_slider, c3_slider, units_html], layout=Layout(width='230px', min_width='230px', flex='0 0 230px', align_items='flex-start', padding='2px 0px 0px 4px', overflow='hidden'))

# ------------------------------------------------------------
# 10. FILTER TYPE ROW
# ------------------------------------------------------------

filter_controls = HBox([filter_type_label, filter_selector], layout=Layout(width='100%', height='30px', align_items='center', justify_content='flex-start', column_gap='48px', margin='0px 0px 4px 0px', padding='0px'))

# ------------------------------------------------------------
# 11. RIGHT COLUMN
# ------------------------------------------------------------

right_column = VBox([filter_controls, magnitude_output, phase_output, info_output], layout=Layout(width='auto', min_width='0px', flex='1 1 auto', align_items='flex-start', overflow='hidden'))

# ------------------------------------------------------------
# 12. MAIN AREA
# ------------------------------------------------------------

main_area = HBox([control_column, right_column], layout=Layout(width='100%', max_width='100%', align_items='flex-start', justify_content='flex-start', overflow='hidden'))

# ------------------------------------------------------------
# 13. FINAL DISPLAY
# ------------------------------------------------------------

display(radio_style)
display(description)
display(main_area)
display(interactive_controls)